In [1]:
# --- 1. ENVIRONMENT SETUP BEFORE IMPORTING LIBRARIES ---
import sys
import os
import re
from pathlib import Path

config_dir = os.path.expanduser("~/.itools")
config_file = os.path.join(config_dir, "config.yml")
dynawo_sys_path = None

# Search for Dynawo path
if os.path.exists(config_file):
    with open(config_file, "r") as f:
        content = f.read()
        match = re.search(r'homeDir:\s*["\']?([^"\'\n]+)["\']?', content)
        if match:
            dynawo_sys_path = match.group(1)

if not dynawo_sys_path or not os.path.exists(dynawo_sys_path):
    raise FileNotFoundError("Error! Valid Dynawo installation not found.")

os.environ["DYNAWO_SYS_PATH"] = dynawo_sys_path
current_dir = os.getcwd()
target_config = os.path.join(config_dir, "config.yml")
template_config = "./om_powsybl_data/config.yml"

# Overwrite configuration with the template
if os.path.exists(template_config):
    with open(template_config, "r") as f:
        template_content = f.read()

    template_content = template_content.replace('"WORKING_DIR/dynawo"', f'"{dynawo_sys_path}"')
    template_content = template_content.replace("WORKING_DIR", current_dir)

    os.makedirs(config_dir, exist_ok=True)
    with open(target_config, "w") as f:
        f.write(template_content)

In [ ]:
# --- 2. IMPORT OF EXTERNAL LIBRARIES AND CONFIGURATION ---
from python_pypowsybl.Code.core.visualizer import NetworkVisualizer
from python_pypowsybl.Code.core.mo_topology import MoTopologyToolkit
from python_pypowsybl.Code.core.powerflow import PowerFlowRunner
from python_pypowsybl.Code.core.comparison import LoadFlowComparator
from python_pypowsybl.Code.core.model_linker import link_models
from python_pypowsybl.Code.core.parameter_generator import DynamicParameterGenerator
from python_pypowsybl.Code.core.cli_runner import DynawoCLIRunner
from python_pypowsybl.Code.core.init_generator import ModelicaInitGenerator
import pypowsybl.dynamic as dyn
from IPython.display import SVG, display, HTML
import matplotlib.pyplot as plt

# Add Code directory to path
scripts_path = os.path.join(current_dir, "../Code")
if scripts_path not in sys.path:
    sys.path.append(scripts_path)

# Base network configurations
SOURCE_DIR = "./om_powsybl_data/SimpleCases/"
MODEL_NAME = "Dynawo.Examples.IEEE57.TestCases.IEEE57NoEvent"
DYNAWO_PKG = "/home/guiu/Projects/Dynawo/nightly/dynawo/ddb/Dynawo/package.mo"
LOCAL_FILES = ["IEEE57NoEvent.mo", "IEEE57Base.mo"]
STR_NAME = "IEEE57"
EXPORT_FOLDER = f"grid_export_{STR_NAME}"

os.makedirs(EXPORT_FOLDER, exist_ok=True)
print("Environment configured and Toolkit loaded successfully.")

'opf' extra dependencies are not installed, some features will not be available


Environment configured and Toolkit loaded successfully.


In [3]:
# --- 3. INSTANTIATE TOOLKIT AND EXECUTE PARSING ---
toolkit = MoTopologyToolkit(SOURCE_DIR, MODEL_NAME, DYNAWO_PKG, LOCAL_FILES)

# Parse the electrical data from the Modelica files and generate a structured representation
parsed_data = toolkit.parse_electrical_data()

# Export for verification into the centralized folder
json_filepath = os.path.join(EXPORT_FOLDER, f"grid_export_{STR_NAME}.json")
toolkit.export_to_standard_json(parsed_data, json_filepath)

# Import for verification
parsed_data = toolkit.import_from_standard_json(json_filepath)

print(f"Parsing successful. Found {len(parsed_data['generators'])} generators.")

[OMC log for 'sendExpression(expr=checkModel(Dynawo.Examples.IEEE57.TestCases.IEEE57NoEvent), parsed=True)']: [translation:error:3] Class Dynawo.Electrical.Transformers.TransformersFixedTap.TransformerFixedRatio not found in scope IEEE57Base.
Initialization CRITICAL FAILURE: OMC error occurred for 'sendExpression(expr=checkModel(Dynawo.Examples.IEEE57.TestCases.IEEE57NoEvent), parsed=True):
00: [translation:error:3] [/home/guiu/Projects/dynawo-notebooks/src/python_pypowsybl/Notebooks/om_powsybl_data/SimpleCases/IEEE57Base.mo:false:213:3:213:205] Class Dynawo.Electrical.Transformers.TransformersFixedTap.TransformerFixedRatio not found in scope IEEE57Base.


OMSessionException: OMC error occurred for 'sendExpression(expr=checkModel(Dynawo.Examples.IEEE57.TestCases.IEEE57NoEvent), parsed=True):
00: [translation:error:3] [/home/guiu/Projects/dynawo-notebooks/src/python_pypowsybl/Notebooks/om_powsybl_data/SimpleCases/IEEE57Base.mo:false:213:3:213:205] Class Dynawo.Electrical.Transformers.TransformersFixedTap.TransformerFixedRatio not found in scope IEEE57Base.

In [ ]:
# --- 4. BUILD POWSYBL NETWORK ---
parsed_data["source_xiidm"] = Path("om_powsybl_data/Base_Case.xiidm").resolve()
# The slack bus is determined automatically by pattern matching the names found in the slack_bus_mapping JSON.
# If no pattern matches, the first bus in the network is defined.
network = toolkit.build_powsybl_network(parsed_data)

print("Network constructed successfully.")
print(f"- Buses: {len(network.get_buses())}")
print(f"- Lines: {len(network.get_lines())}")
print(f"- Generators: {len(network.get_generators())}")
print(f"- Loads: {len(network.get_loads())}")
print(f"- Shunts: {len(network.get_shunt_compensators())}")
print(f"- Transformers: {len(network.get_2_windings_transformers())}")

In [ ]:
# --- 5. NETWORK VISUALIZATION ---
print("Rendering Network Diagrams...")
diagrams = NetworkVisualizer.generate_full_system_diagrams(network)

# 1. Display Macro View
if "network_area" in diagrams:
    display(HTML("<h3>Macro View: Network Area (Lines & Substations)</h3>"))
    display(SVG(str(diagrams["network_area"])))

# 2. Display Micro Views (Generators, Loads, etc.)
display(HTML("<h3>Micro View: Substations Details (Generators visible)</h3>"))
for name, svg in diagrams.items():
    if name != "network_area":
        display(HTML(f"<b>Substation: {name}</b>"))
        display(SVG(str(svg)))

In [ ]:
# --- 6. AC LOAD FLOW EXECUTION AND RESULTS ---
print("Starting AC Load Flow analysis...")

# 1. Execute the Load Flow (logs will be saved to EXPORT_FOLDER if it diverges)
is_converged = PowerFlowRunner.run_ac_loadflow(network, export_path=EXPORT_FOLDER)

if is_converged:
    print("\nSUCCESS: Load flow CONVERGED!")

    # 2. Extract Bus Data (Voltages and Angles)
    print("\n--- BUS RESULTS (Voltages & Angles) ---")
    try:
        buses_df = network.get_buses()
        # In newer versions, we focus on voltage magnitude and angle
        # P and Q injections are viewed at the equipment level or via 'v_mag'/'v_angle'
        columns_buses = ["v_mag", "v_angle"]
        display(buses_df[columns_buses])
    except Exception as e:
        print(f"Could not retrieve Bus results: {e}")

    # 3. Extract Generator Data (To see P and Q output)
    print("\n--- GENERATOR RESULTS (Active & Reactive Power) ---")
    try:
        gens_df = network.get_generators()
        # 'p' and 'q' are the calculated values after loadflow
        columns_gens = ["bus_id", "p", "q", "target_p", "target_v"]
        display(gens_df[columns_gens])
    except Exception as e:
        print(f"Could not retrieve Generator results: {e}")

    # 4. Extract Line Data
    print("\n--- LINE RESULTS (Power Flows & Currents) ---")
    try:
        lines_df = network.get_lines()
        columns_lines = ["bus1_id", "bus2_id", "p1", "q1", "p2", "q2", "i1", "i2"]
        display(lines_df[columns_lines])
    except Exception as e:
        print(f"Could not retrieve Line results: {e}")
else:
    print("\nFAILED: Load flow did NOT converge.")

# Save network to the centralized folder
toolkit.save_powsybl_network(network, export_path=EXPORT_FOLDER)

In [ ]:
# --- 7. CROSS-VALIDATION WITH OPENMODELICA ---
print("Running Ground Truth Validation...")
print("1. Compiling and simulating the Modelica model via OMC...")

INITIALIZATION_MO = "IEEE57_initialized"

toolkit_initialized = MoTopologyToolkit(
    SOURCE_DIR, INITIALIZATION_MO, DYNAWO_PKG, ["IEEE57NoEvent.mo", "IEEE57Base.mo", "IEEE57_initialized.mo"]
)

comparison_df = LoadFlowComparator.compare_voltages(
    network=network,
    connector=toolkit_initialized.connector,
    model_name=INITIALIZATION_MO,
    parsed_data=parsed_data,
)

if not comparison_df.empty:
    print("\nSUCCESS: Validation completed! Showing top discrepancies (sorted by Δ V):")

    # Highlight cells with an error larger than 0.001 pu (0.1%)
    def highlight_errors(val):
        color = "orange" if isinstance(val, (int, float)) and val > 0.001 else "black"
        return f"color: {color}"

    # Apply style to error columns
    styled_df = comparison_df.style.map(
        highlight_errors, subset=["Δ V (pu)", "Δ Theta (deg)"]
    ).format("{:.4f}", na_rep="N/A")

    display(styled_df)

    # Print summary metrics
    mean_v_err = comparison_df["Δ V (pu)"].mean()
    mean_th_err = comparison_df["Δ Theta (deg)"].mean()
    print(f"\n--- Benchmark Summary ---")
    print(f"Average Voltage Mismatch: {mean_v_err:.6f} pu")
    print(f"Average Angle Mismatch:   {mean_th_err:.6f} deg")
else:
    print("Validation failed. Could not retrieve comparison data.")

In [ ]:
# --- 8. DYNAMIC MODEL LINKING AND AUDIT ---
print("Resolving Modelica equations to IIDM static components...\n")

mapping, linked_registry = link_models(network, parsed_data)

if mapping is not None:
    print("=== DYNAMICALLY LINKED MODELS REGISTRY ===")
    for category, df in linked_registry.items():
        print(f"\n--- {category.upper()} ---")
        # Display only the relevant routing columns for clarity
        print(df.to_string(index=False))
else:
    print("CRITICAL ERROR: Model linking failed.")

# Note: This approach is acceptable for now. However, the ideal workflow involves compiling the specific Modelica models alongside their corresponding INIT routines to ensure exact model mapping.

In [ ]:
# --- 9. DYNAMIC PHYSICAL PARAMETERS CONFIGURATION VIA DDB PARSING ---
print("Parsing Dynawo DDB to extract physical parameters for dynamic simulation...")

TARGET_DIR = "om_powsybl_data"
DYNAWO_SYS_PATH = os.environ.get("DYNAWO_SYS_PATH", "/home/guiu/dynawo")

# Run the parameterized generator
DynamicParameterGenerator.generate_parameters(
    parsed_data=parsed_data,
    linked_registry=linked_registry,
    dynawo_path=DYNAWO_SYS_PATH,
    target_dir=TARGET_DIR,
    connector=toolkit.connector,
    root_model_name=MODEL_NAME,
    model_code=toolkit.parser.top_code,
    base_case_file="Base_Case.par",
    network_file="Network.par",
)

print(f"Successfully generated dynamic physical parameters in '{TARGET_DIR}'.")

Steps 11 and 12 are a temporary workaround. Once the `dumpInit` option provided through `dyn.Parameters` is fully operational, these steps can be omitted. 

The current workflow achieves model initialization through the following sequence:

* **Simulation with Debug:** Runs the Dynawo simulation via **PyPowSyBl** with debugging enabled.
* **File Retrieval:** Extracts the generated Dynawo case files from the temporary directory.
* **CLI Re-execution:** Re-runs the simulation via the command line with `dumpInit` set to `true` in the jobs configuration file.

This process yields the necessary files to correctly initialize the Modelica model.

In [ ]:
# --- 10. CONFIGURE DYNAMIC EVENTS AND OUTPUT OBSERVERS ---
print("Configuring topological events and observers...")

events = dyn.EventMapping()
# Add a test short-circuit on busL
events.add_node_fault(static_id="busL", start_time=1.0, fault_time=0.15, r_pu=0.0, x_pu=0.0001)

outputs = dyn.OutputVariableMapping()
# Observe the buses of the inertial grids and the load
outputs.add_standard_model_curves("busIG1", "U_value")
outputs.add_standard_model_curves("busIG2", "U_value")
outputs.add_standard_model_curves("busL", "U_value")

sim_parameters = dyn.Parameters(
    start_time=0.0,
    stop_time=10.0,
    provider_parameters={"dumpInit": "true", "dumpInitDir": EXPORT_FOLDER},
)

print("Events and Observers successfully mapped.")

In [ ]:
# --- 11. EXECUTE DYNAWO TIME-DOMAIN SIMULATION ---
print("Running dynamic simulation via Dynawo engine...")
simulation = dyn.Simulation()

results = simulation.run(
    network,
    model_mapping=mapping,
    event_mapping=events,
    timeseries_mapping=outputs,
    parameters=sim_parameters,
)

if results.status().name == "SUCCESS":
    print("SUCCESS: Time-domain simulation completed.")

    # Extract and Plot State Trajectories
    curves = results.curves()
    plt.figure(figsize=(10, 5))

    # Plot Voltage Responses
    plt.plot(
        curves.index,
        curves["NETWORK_busL_U_value"],
        label="Load Bus (busL)",
        color="red",
    )
    plt.plot(
        curves.index,
        curves["NETWORK_busIG1_U_value"],
        label="Inertial Grid 1",
        linestyle="--",
        color="blue",
    )

    plt.title("Transient Response: IEEE57 Interarea Oscillations")
    plt.xlabel("Time (s)")
    plt.ylabel("Voltage (PU)")
    plt.grid(True)
    plt.legend()
    plt.show()
else:
    print(f"CRITICAL ERROR: Simulation failed. Status: {results.status_text()}")

In [ ]:
# --- 12. COPY FILES, MODIFY JOBS, AND RUN DYNAWO VIA CLI --- (TEMP until dumpIniti option is fully integrated in the CLI)
# Execute the externalized CLI runner script to copy from /tmp and run
DynawoCLIRunner.copy_and_run(export_folder=EXPORT_FOLDER, dynawo_sys_path=DYNAWO_SYS_PATH)

In [ ]:
# --- 13. PARSE DUMPINIT AND GENERATE INITIALIZED MODELICA FILE ---
print("Extracting dump values and building initialized Modelica file...")

target_mo_path = ModelicaInitGenerator.generate_initialized_model(
    export_folder=EXPORT_FOLDER, source_dir=SOURCE_DIR, model_name=MODEL_NAME, network=network
)

if target_mo_path:
    print(f"\nInitialization completed successfully!\nModel saved at: {target_mo_path}")
else:
    print("\nInitialization failed. Check the logs for details.")